In [1]:
import numpy as np
import datetime

from train import train_model as tm
from predictor import predictor as pr
from cell_tracking import tracker as ct

import pandas as pd
import numpy as np
from convert2CTCFormat import lineage_mask

from multiprocessing import cpu_count
import hydra
from hydra.utils import to_absolute_path as abs_path
from hydra import compose, initialize
from omegaconf import OmegaConf

## Learn representation of cell movement

In [5]:
now = datetime.datetime.now()
print('START!', now)
with initialize(version_base='1.3', config_path="config"):
    cfg = compose(config_name="tracker")
    tm(cfg)
    now = datetime.datetime.now()
    print('train DONE!', now)

START! 2026-09-04 20:16:20.407668
please check the configures:

path: ''
dataloader:
  division_detect: true
  start_frame: 0
  num_frame: 200
  itv: 1
  if_crop: false
  tile_num: 25
  pred_tile_num: 4
  overlap: 32
train:
  epochs: 100
  batch_size: 2
  lr: 0.0001
  train_load: false
  load: CP.pth
  val: false
track:
  max_movenment: 50
  load_track: false
  track_file: track_linear_solver.csv
  centroid_file: 2025-09-22-centroid.npy
  method: linear_solver
  run_num: 3
  last_itv: 20
  division: true
  post_pro: true
  min_length: 1
  prune_leaf: true
  merge: true
  jitter_thr: 0.6
  div_interval: 30
  nearest: false



INFO: Using device cuda
INFO: Creating dataset with 92 examples
INFO: Starting training:
        Epochs:          100
        Batch size:      2
        Learning rate:   0.0001
        Training size:   91
        Checkpoints:     result\checkpoint
        Device:          cuda
        Interval:        1
        Optimizer:       Adam
    
Epoch 1/100: 100%|█████████████████████████████████████████████████| 91/91 [02:43<00:00,  1.79s/img, loss (batch)=32.7]
INFO: Checkpoint 1 saved !
Epoch 2/100: 100%|█████████████████████████████████████████████████| 91/91 [02:34<00:00,  1.70s/img, loss (batch)=15.2]
INFO: Checkpoint 2 saved !
Epoch 3/100: 100%|█████████████████████████████████████████████████| 91/91 [02:28<00:00,  1.63s/img, loss (batch)=9.28]
INFO: Checkpoint 3 saved !
Epoch 4/100: 100%|█████████████████████████████████████████████████| 91/91 [02:30<00:00,  1.65s/img, loss (batch)=6.58]
INFO: Checkpoint 4 saved !
Epoch 5/100: 100%|██████████████████████████████████████████████████| 91

train DONE! 2026-09-04 22:51:02.901121


## predict movement field

In [6]:
now = datetime.datetime.now()
print('START!', now)
with initialize(version_base='1.3', config_path="config"):
    cfg = compose(config_name="tracker")
    pr(cfg)
    now = datetime.datetime.now()
    print('predict DONE!', now)

START! 2026-09-04 22:51:02.924201
please check the configures:

path: ''
dataloader:
  division_detect: true
  start_frame: 0
  num_frame: 200
  itv: 1
  if_crop: false
  tile_num: 25
  pred_tile_num: 4
  overlap: 32
train:
  epochs: 100
  batch_size: 2
  lr: 0.0001
  train_load: false
  load: CP.pth
  val: false
track:
  max_movenment: 50
  load_track: false
  track_file: track_linear_solver.csv
  centroid_file: 2025-09-22-centroid.npy
  method: linear_solver
  run_num: 3
  last_itv: 20
  division: true
  post_pro: true
  min_length: 1
  prune_leaf: true
  merge: true
  jitter_thr: 0.6
  div_interval: 30
  nearest: false



INFO: Using device cuda
INFO: Model loaded from result\checkpoint\CP_epoch99.pth
INFO: Creating dataset with 92 examples
                                                                                                                       

predict DONE! 2026-09-04 22:51:17.266205


## Bayesian Estimation and cell tracking

In [7]:
now = datetime.datetime.now()
print('START!', now)
with initialize(version_base='1.3', config_path="config"):
    cfg = compose(config_name="tracker")
    ct(cfg)
    now = datetime.datetime.now()
    print('track DONE!', now)

START! 2026-09-04 22:51:17.280077
please check the configures:

path: ''
dataloader:
  division_detect: true
  start_frame: 0
  num_frame: 200
  itv: 1
  if_crop: false
  tile_num: 25
  pred_tile_num: 4
  overlap: 32
train:
  epochs: 100
  batch_size: 2
  lr: 0.0001
  train_load: false
  load: CP.pth
  val: false
track:
  max_movenment: 50
  load_track: false
  track_file: track_linear_solver.csv
  centroid_file: 2025-09-22-centroid.npy
  method: linear_solver
  run_num: 3
  last_itv: 20
  division: true
  post_pro: true
  min_length: 1
  prune_leaf: true
  merge: true
  jitter_thr: 0.6
  div_interval: 30
  nearest: false

------all running ended-------

2026-09-04 20:11:1592-frame time cost: 33 s
track DONE! 2026-09-04 22:51:50.791073


## save as CTC format and a gif file

In [2]:
mask_dir = r"H:\CellTrack\dataset\Fluo-N2DL-HELA\01_ST\SEG"
track_dir = r"C:\Users\Min Lab\Desktop\xiao\LED\result\2026-09-04track_results.csv"
centroid = r"C:\Users\Min Lab\Desktop\xiao\LED\result\2026-09-04-centroid.npy"

tracks = pd.read_csv(track_dir).to_numpy()
cnt = np.load(centroid, allow_pickle=True)
lineage_mask(mask_dir, tracks, cnt, save_path=r"H:\CellTrack\dataset\Fluo-N2DL-HELA", res_path='RES1')
lineage_mask(mask_dir, tracks, cnt, save_path=r"H:\CellTrack\dataset\Fluo-N2DL-HELA", res_path='RES1', saveRGB=True)

GIF 已保存为：track_mask.gif
